# 02 — Run PMAE and ReMasker

Select a dataset, generate (or reuse) a missingness configuration, train PMAE and ReMasker, and evaluate only the artificially hidden entries. The same workflow is available from the command line.

In [ ]:
from pathlib import Path
import sys
import json

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p / "run_experiment.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or its notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
from pmae_tabular.experiments import run_experiment

## Experiment settings

The defaults below are a proper experiment configuration. For a quick code check, reduce `EPOCHS` to 2; do not report smoke-test metrics as model results.

In [ ]:
DATASET = "adult"
MECHANISM = "MCAR"          # MCAR / MAR / MNAR
COLUMN_FRACTION = 0.30      # fraction of columns selected for missingness
MISSING_RATE = 0.50         # missing fraction inside selected columns
SEED = 42
EPOCHS = 300
BATCH_SIZE = 128
DEVICE = "auto"             # auto / cpu / cuda
MODELS = ("pmae", "remasker")

# Original PMAE experiment-style architecture settings
EMBED_DIM = 32
DEPTH = 4
DECODER_DEPTH = 2
NUM_HEADS = 4
MLP_RATIO = 4.0

## Run the experiment

In [ ]:
results = run_experiment(
    DATASET, ROOT, mechanism=MECHANISM,
    column_fraction=COLUMN_FRACTION, missing_rate=MISSING_RATE,
    seed=SEED, epochs=EPOCHS, batch_size=BATCH_SIZE,
    device=DEVICE, models=MODELS,
    embed_dim=EMBED_DIM, depth=DEPTH,
    decoder_depth=DECODER_DEPTH, num_heads=NUM_HEADS, mlp_ratio=MLP_RATIO,
)
print(json.dumps(results, indent=2, default=str))

## Results location

The run writes imputed CSV files, training histories, model state dictionaries, experiment settings, and `metrics.json` below `results/<dataset>/...`. Record every configuration when comparing datasets or missingness mechanisms.

In [ ]:
from pathlib import Path
result_dir = (ROOT / "results" / DATASET / MECHANISM
              / f"columns_{COLUMN_FRACTION:.2f}"
              / f"rate_{MISSING_RATE:.2f}" / f"seed_{SEED}")
print(result_dir)
print([p.name for p in result_dir.iterdir()])